<a href="https://colab.research.google.com/github/clarencechiang0411/clarencechiang0411.github.io/blob/master/ChatGPT_%E8%AA%9E%E9%9F%B3%E8%BD%89%E6%96%87%E5%AD%9720251122.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# Whisper 語音轉文字
# ============================================
# 步驟：
# 1. 在 Colab 設定 GPU：T4
# 2. 若要用雲端硬碟，請將 USE_GOOGLE_DRIVE 改為 True
# 3. 設定你的檔案路徑 INPUT_FILE_PATH
# 4. 執行就會輸出 TXT 與 SRT 字幕

# ---------- 基本設定 ----------
USE_GOOGLE_DRIVE = True  # 是否使用 Google Drive

# <<< 請改成你的檔案路徑 >>>
INPUT_FILE_PATH = "/content/250214_1605.mp3"

# <<< 逐字稿輸出位置 >>>
OUTPUT_FOLDER = "/content/drive/MyDrive/whisper_project/transcripts"

MODEL_NAME = "large"  # small 較快、medium 較準 large 最準確也最慢
LANG = "zh"           # 中文

# ---------- 安裝套件 ----------
!pip install -q openai-whisper torch librosa soundfile moviepy tqdm

import torch, librosa, soundfile as sf, whisper, os
from moviepy.editor import VideoFileClip
from pathlib import Path
from tqdm.notebook import tqdm

# ---------- 掛載 Google Drive ----------
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# ---------- 檢查 GPU ----------
if torch.cuda.is_available():
    print("使用 GPU：", torch.cuda.get_device_name(0))
else:
    print("目前沒有使用 GPU，執行會變慢喔！")

# ---------- 載入 Whisper 模型 ----------
print("載入 Whisper 模型中...")
model = whisper.load_model(MODEL_NAME).to("cuda" if torch.cuda.is_available() else "cpu")

# ---------- 準備音訊（如果是影片就抽音訊） ----------
input_path = Path(INPUT_FILE_PATH)

if input_path.suffix.lower() in [".mp4", ".mov", ".avi", ".mkv"]:
    print("偵測到影片，正在抽取音訊...")
    clip = VideoFileClip(str(input_path))
    audio_path = input_path.parent / "temp_audio.wav"
    clip.audio.write_audiofile(str(audio_path), codec="pcm_s16le")
    clip.close()
else:
    audio_path = input_path

# ---------- Whisper 轉文字 ----------
print("開始轉錄...")
result = model.transcribe(str(audio_path), language=LANG)

text = result["text"]

# ---------- 輸出資料夾 ----------
output_dir = Path(OUTPUT_FOLDER)
output_dir.mkdir(parents=True, exist_ok=True)

# ---------- 輸出 TXT ----------
txt_path = output_dir / f"{input_path.stem}.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(text)

# ---------- 輸出 SRT ----------
def to_srt(segments):
    srt = ""
    for i, seg in enumerate(segments):
        start = seg["start"]
        end = seg["end"]
        text = seg["text"].strip()

        def format_time(t):
            ms = int(t * 1000)
            s, ms = divmod(ms, 1000)
            m, s = divmod(s, 60)
            h, m = divmod(m, 60)
            return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

        srt += f"{i+1}\n"
        srt += f"{format_time(start)} --> {format_time(end)}\n"
        srt += text + "\n\n"
    return srt

srt_content = to_srt(result["segments"])

srt_path = output_dir / f"{input_path.stem}.srt"
with open(srt_path, "w", encoding="utf-8") as f:
    f.write(srt_content)

print("轉錄完成！")
print("TXT 輸出：", txt_path)
print("SRT 輸出：", srt_path)

# 清理暫存音訊
if audio_path.name == "temp_audio.wav":
    os.remove(audio_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
目前沒有使用 GPU，執行會變慢喔！
載入 Whisper 模型中...


100%|█████████████████████████████████████| 2.88G/2.88G [00:33<00:00, 92.3MiB/s]
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



開始轉錄...
